# Data Loading

In [261]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 0 ns
Wall time: 15.6 ms


In [262]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


# Training with Logistic Regression

Simple PyTorch logistic regression with a soft‑cross‑entropy loss.


### Model

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [4]:
# --- 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by a temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

In [5]:
# --- 2) sklearn‐style Torch Logistic Regression -----------------------

class TorchLogisticClassifier(BaseEstimator, ClassifierMixin):
    """
    A sklearn‐compatible PyTorch logistic regression with L1/L2/Elastic-Net
    and soft‐CE temperature scaling.
    """
    def __init__(
        self,
        penalty: str=None,       # none,'l1', 'l2', or 'elasticnet'
        alpha: float = 1e-3,       # regularization strength
        beta: float = 0.5,     # for elastic-net
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.penalty = penalty
        self.alpha = alpha
        self.beta = beta
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.temperature = temperature
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):
        # Validate inputs
        X, y = check_X_y(X, y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # Device
        self.device_ = torch.device(
            self.device or ('cuda' if torch.cuda.is_available() else 'cpu')
        )
        torch.manual_seed(self.random_state)

        # Scale features
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)

        # Map classes to indices
        self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
        y_idx = np.vectorize(self.class_to_idx_.get)(y)

        # Build DataLoader
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        y_tensor = torch.tensor(y_idx,    dtype=torch.long,   device=self.device_)
        dataset  = TensorDataset(X_tensor, y_tensor)
        loader   = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # Model
        self.model_ = nn.Linear(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr)

        # Training loop
        for epoch in range(1, self.epochs + 1):
            self.model_.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss   = criterion(logits, yb)
                # print('loss before regularization', loss)

                # Regularization
                l1 = sum(p.abs().sum() for p in self.model_.parameters())
                l2 = sum(p.pow(2).sum() for p in self.model_.parameters())
                # print('l1, l2, Loss', l1, l2, loss)
                if self.penalty == 'l1':
                    loss = loss + self.alpha * l1
                    # print('l1 loss', loss, l1)
                elif self.penalty == 'l2':
                    loss = loss + self.alpha * l2
                    # print('l2 loss', loss, l2)
                elif self.penalty == 'elasticnet':
                    loss = loss + self.alpha * (self.beta * l1 + (1 - self.beta) * l2)
                    # print('elasticnet', loss)
                else:
                    loss = loss
                    # print('loss', loss)


                loss.backward()
                optimizer.step()

        return self

    def predict(self, X):
        check_is_fitted(self, ['model_', 'scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds  = logits.argmax(dim=1).cpu().numpy()
        idx_to_class = {i:c for c,i in self.class_to_idx_.items()}
        return np.vectorize(idx_to_class.get)(preds)

    def predict_proba(self, X):
        check_is_fitted(self, ['model_', 'scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            probs  = F.softmax(logits / self.temperature, dim=1).cpu().numpy()
        return probs

In [ ]:
import pandas as pd
import numpy as np

def torch_logistic_feature_importances(clf, feature_names):
    """
    Given a fitted TorchLogisticClassifier `clf` and the list of feature names,
    returns a DataFrame of absolute average weights per feature, sorted descending.
    """
    # Grab the learned weight tensor (shape: n_classes x n_features)
    W = clf.model_.weight.detach().cpu().numpy()   # ndarray of shape (C, F)
    
    # Compute per-Coefficients (Linear Models): e.g. mean of absolute weights across classes
    coefficents = np.mean(np.abs(W), axis=0)       # shape (F,)
    
    # Build DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficents': coefficents
    }).sort_values('coefficents', ascending=False).reset_index(drop=True)
    
    return coef_df



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

Normal training using train-test-split with 20% of data as test data

In [264]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization

In [274]:
# Instantiate and train
clf_strat_y = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.75,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=32, 
    epochs=200,
)
clf_strat_y.fit(X_train, y_train)



TorchLogisticClassifier(alpha=0, batch_size=32, epochs=200, lr=0.75)

Coefficients (Linear Models)

In [278]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0           d_avg     2.373612
1   shell_avgSize     1.348984
2        core_min     1.322395
3        core_max     1.168570
4   shell_maxSize     0.963466
5               m     0.943921
6               n     0.745482
7           d_max     0.318226
8       shell_num     0.253462
9   shell_minSize     0.220079
10        k_value     0.200505
11  shell_stdSize     0.066163


#### Inference on 20% dataset

In [276]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.48%

Precision by Algorithm:
  UMKP: 100.00% (6 predictions)
  AltRB: 57.58% (33 predictions)
  kPEX: 62.15% (214 predictions)
  Pivot_Br: 13.79% (58 predictions)
  S_Br: 42.20% (109 predictions)

Recall by Algorithm:
  UMKP: 34.69% (17/49 cases)
  AltRB: 56.67% (34/60 cases)
  kPEX: 68.84% (148/215 cases)
  Pivot_Br: 59.35% (92/155 cases)
  S_Br: 67.79% (101/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 64397.30
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.334
  Avg Overhead/case ((pred - opt) / opt)    = 63.29%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 208 wrong predictions with time metrics



#### Re-evalute on entire dataset

In [277]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.84%

Precision by Algorithm:
  UMKP: 61.11% (18 predictions)
  AltRB: 52.11% (71 predictions)
  kPEX: 55.83% (369 predictions)
  Pivot_Br: 14.04% (114 predictions)
  S_Br: 45.74% (188 predictions)

Recall by Algorithm:
  UMKP: 33.00% (33/100 cases)
  AltRB: 51.72% (60/116 cases)
  kPEX: 63.64% (231/363 cases)
  Pivot_Br: 57.96% (131/226 cases)
  S_Br: 63.28% (162/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 107707.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.610
  Avg Overhead/case ((pred - opt) / opt)    = 242.86%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 404 wrong predictions with time 

#### LR with L1 regularization

In [283]:
# Instantiate and train
clf_l1_strat_y = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.8,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=32, 
    epochs=200,
)
clf_l1_strat_y.fit(X_train, y_train)



TorchLogisticClassifier(batch_size=32, epochs=200, lr=0.8, penalty='l1')

Coefficients (Linear Models)

In [284]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.583658
1           d_max     0.321965
2       shell_num     0.263834
3         k_value     0.175722
4           d_avg     0.097465
5        core_max     0.086757
6        core_min     0.083499
7               n     0.077024
8   shell_maxSize     0.076959
9   shell_minSize     0.076464
10  shell_avgSize     0.068626
11  shell_stdSize     0.056938


#### Inference on 20% dataset

In [285]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.33%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 55.56% (36 predictions)
  kPEX: 60.49% (205 predictions)
  Pivot_Br: 12.82% (39 predictions)
  S_Br: 38.57% (140 predictions)

Recall by Algorithm:
  UMKP: 20.41% (10/49 cases)
  AltRB: 48.33% (29/60 cases)
  kPEX: 66.05% (142/215 cases)
  Pivot_Br: 56.77% (88/155 cases)
  S_Br: 71.14% (106/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 75307.20
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.560
  Avg Overhead/case ((pred - opt) / opt)    = 210.46%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 217 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [286]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 45.79%

Precision by Algorithm:
  UMKP: 44.44% (9 predictions)
  AltRB: 55.84% (77 predictions)
  kPEX: 55.20% (346 predictions)
  Pivot_Br: 14.74% (95 predictions)
  S_Br: 41.20% (233 predictions)

Recall by Algorithm:
  UMKP: 28.00% (28/100 cases)
  AltRB: 52.59% (61/116 cases)
  kPEX: 60.33% (219/363 cases)
  Pivot_Br: 56.64% (128/226 cases)
  S_Br: 66.02% (169/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 144271.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 2.157
  Avg Overhead/case ((pred - opt) / opt)    = 478.57%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 412 wrong predictions with time metrics

T

#### LR with L2 regularization

In [291]:
# Instantiate and train
clf_l2_strat_y = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.75,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    batch_size=32,       
    epochs=200,
)
clf_l2_strat_y.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=32, epochs=200, lr=0.75, penalty='l2')

Coefficients (Linear Models)

In [292]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.636778
1           d_max     0.291387
2           d_avg     0.278878
3       shell_num     0.250907
4        core_min     0.193452
5        core_max     0.189671
6         k_value     0.172645
7   shell_maxSize     0.091513
8               n     0.091513
9   shell_minSize     0.091365
10  shell_avgSize     0.073356
11  shell_stdSize     0.070435


#### Inference on 20% dataset

In [293]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.57%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 55.56% (36 predictions)
  kPEX: 60.89% (202 predictions)
  Pivot_Br: 11.63% (43 predictions)
  S_Br: 40.29% (139 predictions)

Recall by Algorithm:
  UMKP: 20.41% (10/49 cases)
  AltRB: 48.33% (29/60 cases)
  kPEX: 65.58% (141/215 cases)
  Pivot_Br: 56.77% (88/155 cases)
  S_Br: 72.48% (108/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73659.40
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.526
  Avg Overhead/case ((pred - opt) / opt)    = 164.56%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 216 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [294]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 45.26%

Precision by Algorithm:
  UMKP: 42.86% (7 predictions)
  AltRB: 55.13% (78 predictions)
  kPEX: 54.65% (344 predictions)
  Pivot_Br: 13.21% (106 predictions)
  S_Br: 42.67% (225 predictions)

Recall by Algorithm:
  UMKP: 27.00% (27/100 cases)
  AltRB: 51.72% (60/116 cases)
  kPEX: 59.50% (216/363 cases)
  Pivot_Br: 56.64% (128/226 cases)
  S_Br: 66.02% (169/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 134878.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 2.017
  Avg Overhead/case ((pred - opt) / opt)    = 415.77%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 416 wrong predictions with time metrics



#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [298]:
# Instantiate and train
clf_elasticnet_strat_y = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.7,         # regularization strength of L1 in usint elasticnet
    lr=0.75,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.     
    batch_size=32,  
    epochs=200,
)
clf_elasticnet_strat_y.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=32, beta=0.7, epochs=200, lr=0.75,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [299]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.594800
1           d_max     0.298714
2       shell_num     0.255772
3         k_value     0.170749
4           d_avg     0.135008
5        core_max     0.109140
6        core_min     0.108267
7   shell_maxSize     0.082393
8               n     0.082337
9   shell_minSize     0.080571
10  shell_avgSize     0.068830
11  shell_stdSize     0.062885


#### Inference on 20% dataset

In [300]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.33%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 55.56% (36 predictions)
  kPEX: 61.08% (203 predictions)
  Pivot_Br: 11.63% (43 predictions)
  S_Br: 39.13% (138 predictions)

Recall by Algorithm:
  UMKP: 20.41% (10/49 cases)
  AltRB: 48.33% (29/60 cases)
  kPEX: 66.05% (142/215 cases)
  Pivot_Br: 56.77% (88/155 cases)
  S_Br: 71.14% (106/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 75310.80
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.560
  Avg Overhead/case ((pred - opt) / opt)    = 210.37%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 217 wrong predictions with time metrics

Top 5 

#### Re-evalute on entire dataset

In [301]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 45.26%

Precision by Algorithm:
  UMKP: 50.00% (8 predictions)
  AltRB: 54.55% (77 predictions)
  kPEX: 54.81% (343 predictions)
  Pivot_Br: 13.46% (104 predictions)
  S_Br: 42.11% (228 predictions)

Recall by Algorithm:
  UMKP: 27.00% (27/100 cases)
  AltRB: 51.72% (60/116 cases)
  kPEX: 59.50% (216/363 cases)
  Pivot_Br: 56.64% (128/226 cases)
  S_Br: 66.02% (169/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 138902.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 2.077
  Avg Overhead/case ((pred - opt) / opt)    = 466.19%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 416 wrong predictions with time metric

### **Balanced train-test split based on both Y and k**

In [302]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df,test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )
# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")



Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization

In [303]:
# Instantiate and train
clf_strat_y_K = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.75,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=200,
)
clf_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(alpha=0, batch_size=128, epochs=200, lr=0.75)

Coefficients (Linear Models)

In [304]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0           d_avg     1.168586
1               m     0.940658
2        core_min     0.712975
3   shell_maxSize     0.572922
4        core_max     0.540974
5           d_max     0.388726
6               n     0.354938
7   shell_avgSize     0.312922
8       shell_num     0.195065
9   shell_minSize     0.159106
10        k_value     0.098148
11  shell_stdSize     0.070369


#### Inference on 20% dataset

In [305]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 55.71%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 65.22% (23 predictions)
  kPEX: 62.03% (316 predictions)
  Pivot_Br: 28.21% (39 predictions)
  S_Br: 30.00% (40 predictions)

Recall by Algorithm:
  UMKP: 35.85% (19/53 cases)
  AltRB: 43.48% (30/69 cases)
  kPEX: 93.27% (208/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 46.58% (68/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59101.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.274
  Avg Overhead/case ((pred - opt) / opt)    = 180.89%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 186 wrong predictions with time metrics

T

#### Re-evalute on entire dataset

In [306]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.66%

Precision by Algorithm:
  UMKP: 50.00% (8 predictions)
  AltRB: 60.78% (51 predictions)
  kPEX: 55.13% (575 predictions)
  Pivot_Br: 22.73% (66 predictions)
  S_Br: 30.00% (60 predictions)

Recall by Algorithm:
  UMKP: 26.00% (26/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 92.56% (336/363 cases)
  Pivot_Br: 58.41% (132/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 107795.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.612
  Avg Overhead/case ((pred - opt) / opt)    = 290.67%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 375 wrong predictions with time m

#### LR with L1 regularization

In [307]:
# Instantiate and train
clf_l1_strat_y_K = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.3,              # Learning rate
    batch_size=128,        
    epochs=200,
)
clf_l1_strat_y_K.fit(X_train, y_train)



TorchLogisticClassifier(batch_size=128, epochs=200, lr=0.3, penalty='l1')

Coefficients (Linear Models)

In [308]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.592567
1           d_max     0.212367
2       shell_num     0.185248
3   shell_maxSize     0.127497
4   shell_minSize     0.101835
5               n     0.070281
6        core_min     0.068339
7         k_value     0.059579
8        core_max     0.039855
9           d_avg     0.038510
10  shell_stdSize     0.037307
11  shell_avgSize     0.009407


#### Inference on 20% dataset

In [309]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 55.95%

Precision by Algorithm:
  UMKP: 14.29% (7 predictions)
  AltRB: 70.00% (20 predictions)
  kPEX: 60.47% (344 predictions)
  Pivot_Br: 20.00% (20 predictions)
  S_Br: 27.59% (29 predictions)

Recall by Algorithm:
  UMKP: 39.62% (21/53 cases)
  AltRB: 40.58% (28/69 cases)
  kPEX: 95.52% (213/223 cases)
  Pivot_Br: 65.81% (102/155 cases)
  S_Br: 48.63% (71/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55769.50
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.202
  Avg Overhead/case ((pred - opt) / opt)    = 136.91%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 185 wrong predictions with time metrics

Top 5 h

#### Re-evalute on entire dataset

In [310]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 51.71%

Precision by Algorithm:
  UMKP: 55.56% (18 predictions)
  AltRB: 59.09% (44 predictions)
  kPEX: 54.95% (626 predictions)
  Pivot_Br: 16.67% (36 predictions)
  S_Br: 19.44% (36 predictions)

Recall by Algorithm:
  UMKP: 30.00% (30/100 cases)
  AltRB: 37.07% (43/116 cases)
  kPEX: 96.42% (350/363 cases)
  Pivot_Br: 58.41% (132/226 cases)
  S_Br: 38.67% (99/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 103329.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.545
  Avg Overhead/case ((pred - opt) / opt)    = 290.09%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 367 wrong predictions with time metrics



#### LR with L2 regularization

In [311]:
# Instantiate and train
clf_l2_strat_y_K = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    batch_size=128,
    epochs=200,
)
clf_l2_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, epochs=200, lr=0.25, penalty='l2')

Coefficients (Linear Models)

In [312]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.631644
1           d_max     0.234807
2           d_avg     0.229416
3       shell_num     0.204650
4        core_min     0.191910
5   shell_maxSize     0.159977
6        core_max     0.141343
7               n     0.106241
8   shell_minSize     0.096677
9         k_value     0.078662
10  shell_stdSize     0.050611
11  shell_avgSize     0.030897


#### Inference on 20% dataset

In [313]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 54.29%

Precision by Algorithm:
  UMKP: 16.67% (6 predictions)
  AltRB: 71.43% (21 predictions)
  kPEX: 60.49% (329 predictions)
  Pivot_Br: 11.76% (34 predictions)
  S_Br: 30.00% (30 predictions)

Recall by Algorithm:
  UMKP: 39.62% (21/53 cases)
  AltRB: 42.03% (29/69 cases)
  kPEX: 91.48% (204/223 cases)
  Pivot_Br: 66.45% (103/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 56167.90
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.210
  Avg Overhead/case ((pred - opt) / opt)    = 137.32%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 192 wrong predictions with time metrics

Top 5 h

#### Re-evalute on entire dataset

In [314]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.61%

Precision by Algorithm:
  UMKP: 53.33% (15 predictions)
  AltRB: 60.87% (46 predictions)
  kPEX: 53.97% (604 predictions)
  Pivot_Br: 12.28% (57 predictions)
  S_Br: 21.05% (38 predictions)

Recall by Algorithm:
  UMKP: 29.00% (29/100 cases)
  AltRB: 37.93% (44/116 cases)
  kPEX: 91.46% (332/363 cases)
  Pivot_Br: 59.29% (134/226 cases)
  S_Br: 38.67% (99/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102380.70
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.531
  Avg Overhead/case ((pred - opt) / opt)    = 281.32%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 383 wrong predictions with time metrics



#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [319]:
# Instantiate and train
clf_elasticnet_strat_y_K = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.7,         # regularization strength of L1 in usint elasticnet
    lr=0.75,              # Learning rate
    batch_size=128,       # Feeding whole data at a time.
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    epochs=200,
)
clf_elasticnet_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, beta=0.7, epochs=200, lr=0.75,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [320]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.590484
1           d_max     0.335707
2       shell_num     0.196391
3   shell_maxSize     0.119690
4               n     0.104765
5           d_avg     0.092050
6        core_min     0.091257
7         k_value     0.080461
8   shell_minSize     0.073701
9        core_max     0.069192
10  shell_stdSize     0.062292
11  shell_avgSize     0.021761


#### Inference on 20% dataset

In [321]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.19%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 61.11% (18 predictions)
  kPEX: 61.56% (333 predictions)
  Pivot_Br: 31.58% (38 predictions)
  S_Br: 27.59% (29 predictions)

Recall by Algorithm:
  UMKP: 33.96% (18/53 cases)
  AltRB: 37.68% (26/69 cases)
  kPEX: 95.96% (214/223 cases)
  Pivot_Br: 65.16% (101/155 cases)
  S_Br: 46.58% (68/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59219.90
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.276
  Avg Overhead/case ((pred - opt) / opt)    = 189.12%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 184 wrong predictions with time metrics

Top 5

#### Re-evalute on entire dataset

In [322]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.26%

Precision by Algorithm:
  UMKP: 50.00% (8 predictions)
  AltRB: 58.54% (41 predictions)
  kPEX: 54.59% (599 predictions)
  Pivot_Br: 21.21% (66 predictions)
  S_Br: 28.26% (46 predictions)

Recall by Algorithm:
  UMKP: 24.00% (24/100 cases)
  AltRB: 35.34% (41/116 cases)
  kPEX: 93.94% (341/363 cases)
  Pivot_Br: 58.41% (132/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 111086.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.661
  Avg Overhead/case ((pred - opt) / opt)    = 350.45%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 378 wrong predictions with time metric

## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


### **Balanced train-test split based on Y**

In [323]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [324]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization


In [325]:
# Instantiate & train
clf_LR_no_penalty_strat_y = TorchLogisticClassifier(
        penalty=None,
        lr=0.01,
        batch_size=32,
        epochs=1000,
    )

clf_LR_no_penalty_strat_y.fit(X_train_, y_train_)

TorchLogisticClassifier(batch_size=32, epochs=1000)

Coefficients (Linear Models)

In [327]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_LR_no_penalty_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.598553
1        core_min     0.299954
2           d_avg     0.284744
3       shell_num     0.263908
4   shell_maxSize     0.247888
5           d_max     0.195789
6   shell_avgSize     0.163559
7         k_value     0.111392
8        core_max     0.109680
9   shell_minSize     0.092945
10              n     0.061123
11  shell_stdSize     0.054092


#### Inference on 20% dataset

In [326]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_LR_no_penalty_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.67%

Precision by Algorithm:
  UMKP: 33.33% (3 predictions)
  AltRB: 74.07% (27 predictions)
  kPEX: 62.96% (297 predictions)
  Pivot_Br: 29.17% (48 predictions)
  S_Br: 35.56% (45 predictions)

Recall by Algorithm:
  UMKP: 41.51% (22/53 cases)
  AltRB: 50.72% (35/69 cases)
  kPEX: 90.58% (202/223 cases)
  Pivot_Br: 65.16% (101/155 cases)
  S_Br: 50.68% (74/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54905.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.183
  Avg Overhead/case ((pred - opt) / opt)    = 133.22%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 182 wrong predictions with time metrics

T

#### Re-evalute on entire dataset

In [328]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_LR_no_penalty_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.92%

Precision by Algorithm:
  UMKP: 52.63% (19 predictions)
  AltRB: 64.58% (48 predictions)
  kPEX: 55.66% (548 predictions)
  Pivot_Br: 26.19% (84 predictions)
  S_Br: 31.15% (61 predictions)

Recall by Algorithm:
  UMKP: 32.00% (32/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 90.08% (327/363 cases)
  Pivot_Br: 59.73% (135/226 cases)
  S_Br: 39.45% (101/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99050.50
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.481
  Avg Overhead/case ((pred - opt) / opt)    = 232.03%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 373 wrong predictions with time met

#### LR with L1 regularization

In [329]:
# Instantiate and train
clf_l1_strat_y = TorchLogisticClassifier(
        penalty='l1',
        alpha=1e-3,
        lr=1e-2,
        batch_size=128,
        epochs=1000,
    )
clf_l1_strat_y.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, epochs=1000, penalty='l1')

Coefficients (Linear Models)

In [330]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.391230
1       shell_num     0.206233
2           d_max     0.115297
3        core_min     0.110313
4   shell_stdSize     0.095252
5   shell_maxSize     0.075224
6           d_avg     0.065593
7         k_value     0.056383
8   shell_minSize     0.048584
9   shell_avgSize     0.025902
10       core_max     0.019912
11              n     0.015847


#### Inference on 20% dataset

In [331]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.71%

Precision by Algorithm:
  UMKP: 50.00% (12 predictions)
  AltRB: 76.47% (17 predictions)
  kPEX: 58.33% (300 predictions)
  Pivot_Br: 16.88% (77 predictions)
  S_Br: 42.86% (14 predictions)

Recall by Algorithm:
  UMKP: 47.17% (25/53 cases)
  AltRB: 39.13% (27/69 cases)
  kPEX: 80.72% (180/223 cases)
  Pivot_Br: 69.68% (108/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 56369.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.215
  Avg Overhead/case ((pred - opt) / opt)    = 137.47%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 207 wrong predictions with time metrics

Top 5 he

#### Re-evalute on entire dataset

In [332]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.84%

Precision by Algorithm:
  UMKP: 59.38% (32 predictions)
  AltRB: 63.33% (30 predictions)
  kPEX: 53.96% (530 predictions)
  Pivot_Br: 18.62% (145 predictions)
  S_Br: 21.74% (23 predictions)

Recall by Algorithm:
  UMKP: 37.00% (37/100 cases)
  AltRB: 31.90% (37/116 cases)
  kPEX: 81.27% (295/363 cases)
  Pivot_Br: 64.16% (145/226 cases)
  S_Br: 39.45% (101/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102641.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.535
  Avg Overhead/case ((pred - opt) / opt)    = 234.85%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 404 wrong predictions with time metrics



#### LR with L2 regularization

In [333]:
# Instantiate and train
clf_l2_strat_y = TorchLogisticClassifier(
        penalty='l2',
        alpha=1e-2,
        beta=0.7,
        lr=1e-2,
        batch_size=128,
        epochs=100,
    )
clf_l2_strat_y.fit(X_train, y_train)


TorchLogisticClassifier(alpha=0.01, batch_size=128, beta=0.7, epochs=100,
                        penalty='l2')

Coefficients (Linear Models)

In [334]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0        core_min     0.136232
1   shell_avgSize     0.131512
2       shell_num     0.127192
3   shell_stdSize     0.122392
4               m     0.118859
5               n     0.107214
6           d_avg     0.106994
7         k_value     0.101776
8   shell_minSize     0.095497
9   shell_maxSize     0.079584
10       core_max     0.065749
11          d_max     0.048722


#### Inference on 20% dataset

In [335]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.62%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 44.12% (34 predictions)
  kPEX: 60.47% (301 predictions)
  Pivot_Br: 22.95% (61 predictions)
  S_Br: 45.45% (22 predictions)

Recall by Algorithm:
  UMKP: 37.74% (20/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 67.74% (105/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 61089.60
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.316
  Avg Overhead/case ((pred - opt) / opt)    = 214.44%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 199 wrong predictions with time metrics

Top 5 heav

#### Re-evalute on entire dataset

In [336]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.55%

Precision by Algorithm:
  UMKP: 63.64% (11 predictions)
  AltRB: 46.15% (65 predictions)
  kPEX: 54.91% (550 predictions)
  Pivot_Br: 21.50% (107 predictions)
  S_Br: 25.93% (27 predictions)

Recall by Algorithm:
  UMKP: 29.00% (29/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 86.23% (313/363 cases)
  Pivot_Br: 61.95% (140/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 104756.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.566
  Avg Overhead/case ((pred - opt) / opt)    = 258.42%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 391 wrong predictions with time metrics



#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [337]:
# Instantiate and train
clf_elasticnet_strat_y = TorchLogisticClassifier(
        penalty='elasticnet',
        alpha=1e-3,
        beta=0.7,
        lr=1e-2,
        batch_size=128, # batch size = X.shape
        epochs=100,
    )
clf_elasticnet_strat_y.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, beta=0.7, epochs=100,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [338]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y, feature_cols)
print(coef_df)

          feature  coefficents
0        core_min     0.162781
1   shell_avgSize     0.162600
2       shell_num     0.136016
3   shell_stdSize     0.132233
4               n     0.128314
5               m     0.126527
6           d_avg     0.125410
7   shell_minSize     0.118283
8         k_value     0.115459
9   shell_maxSize     0.099194
10       core_max     0.076786
11          d_max     0.051268


#### Inference on 20% dataset

In [339]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.38%

Precision by Algorithm:
  UMKP: 0.00% (2 predictions)
  AltRB: 44.12% (34 predictions)
  kPEX: 60.87% (299 predictions)
  Pivot_Br: 20.97% (62 predictions)
  S_Br: 43.48% (23 predictions)

Recall by Algorithm:
  UMKP: 37.74% (20/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 67.10% (104/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 61020.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.315
  Avg Overhead/case ((pred - opt) / opt)    = 212.18%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 200 wrong predictions with time metrics

Top 5 h

#### Re-evalute on entire dataset

In [340]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.29%

Precision by Algorithm:
  UMKP: 63.64% (11 predictions)
  AltRB: 45.31% (64 predictions)
  kPEX: 55.29% (548 predictions)
  Pivot_Br: 19.63% (107 predictions)
  S_Br: 23.33% (30 predictions)

Recall by Algorithm:
  UMKP: 29.00% (29/100 cases)
  AltRB: 40.52% (47/116 cases)
  kPEX: 86.50% (314/363 cases)
  Pivot_Br: 61.06% (138/226 cases)
  S_Br: 39.06% (100/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 106157.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.587
  Avg Overhead/case ((pred - opt) / opt)    = 301.73%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 393 wrong predictions with time metric

In [ ]:
# ## For exporting classification reports to a temp XL file.

# from write_utils import append_report_to_excel

# append_report_to_excel(
#     filename="temp_results.xlsx",
#     heading="Balanced train-test split based on Y ",
#     classif_report_df=classif_df,
#     feat_importances_df=importances,
#     sheet_name="Sheet1"    # or whatever your sheet is called
# )


### **Balanced train-test split based on both Y and k**

In [341]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [342]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train, y_train, weights_train = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train = le.fit_transform(y_train)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### LR without regularization

In [346]:
# Instantiate and train
clf_strat_y_K = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=1000,
)
clf_strat_y_K.fit(X_train, y_train)

TorchLogisticClassifier(alpha=0, batch_size=128, epochs=1000, lr=0.25)

Coefficients (Linear Models)

In [347]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0           d_avg     1.706208
1        core_min     0.984353
2        core_max     0.818618
3   shell_avgSize     0.731164
4   shell_maxSize     0.593695
5               m     0.589565
6               n     0.375709
7           d_max     0.230942
8       shell_num     0.190099
9   shell_minSize     0.179037
10        k_value     0.133226
11  shell_stdSize     0.072736


#### Inference on 20% dataset

In [348]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.62%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 67.86% (28 predictions)
  kPEX: 58.90% (292 predictions)
  Pivot_Br: 20.00% (35 predictions)
  S_Br: 35.38% (65 predictions)

Recall by Algorithm:
  UMKP: 33.96% (18/53 cases)
  AltRB: 47.83% (33/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 51.37% (75/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55020.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.186
  Avg Overhead/case ((pred - opt) / opt)    = 134.50%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 199 wrong predictions with time metrics

T

Metrics written to ./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx


#### Re-evalute on entire dataset

In [349]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/No_penalty/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/No_penalty/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.92%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 56.45% (62 predictions)
  kPEX: 55.29% (539 predictions)
  Pivot_Br: 23.88% (67 predictions)
  S_Br: 41.30% (92 predictions)

Recall by Algorithm:
  UMKP: 23.00% (23/100 cases)
  AltRB: 44.83% (52/116 cases)
  kPEX: 88.98% (323/363 cases)
  Pivot_Br: 58.41% (132/226 cases)
  S_Br: 46.48% (119/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 97558.70
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.459
  Avg Overhead/case ((pred - opt) / opt)    = 234.38%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/No_penalty/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 373 wrong predictions with time met

#### LR with L1 regularization

In [350]:
# Instantiate and train
clf_l1_strat_y_K = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    batch_size=128,        
    epochs=1000,
)
clf_l1_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, epochs=1000, lr=0.25, penalty='l1')

Coefficients (Linear Models)

In [351]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l1_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.340855
1       shell_num     0.193455
2           d_max     0.163610
3   shell_minSize     0.132404
4        core_min     0.102922
5         k_value     0.063169
6           d_avg     0.043792
7   shell_stdSize     0.039189
8               n     0.022302
9   shell_maxSize     0.022287
10       core_max     0.018797
11  shell_avgSize     0.013350


#### Inference on 20% dataset

In [352]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.81%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 55.56% (27 predictions)
  kPEX: 60.40% (303 predictions)
  Pivot_Br: 25.58% (43 predictions)
  S_Br: 36.17% (47 predictions)

Recall by Algorithm:
  UMKP: 32.08% (17/53 cases)
  AltRB: 46.38% (32/69 cases)
  kPEX: 87.89% (196/223 cases)
  Pivot_Br: 63.23% (98/155 cases)
  S_Br: 50.68% (74/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 57454.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.238
  Avg Overhead/case ((pred - opt) / opt)    = 199.15%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 194 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [353]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l1_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.66%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 51.72% (58 predictions)
  kPEX: 56.06% (553 predictions)
  Pivot_Br: 23.17% (82 predictions)
  S_Br: 38.81% (67 predictions)

Recall by Algorithm:
  UMKP: 22.00% (22/100 cases)
  AltRB: 41.38% (48/116 cases)
  kPEX: 90.91% (330/363 cases)
  Pivot_Br: 59.29% (134/226 cases)
  S_Br: 43.75% (112/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102499.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.533
  Avg Overhead/case ((pred - opt) / opt)    = 296.78%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...



Saved 375 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
          dataset  k_value      weight predicted fastest_algorithms  \
646    bag-pubmed        3  662.182989  Pivot_Br             [UMKP]   
491       keller4        5  660.562496      kPEX             [S_Br]   
470   edit-itwiki        5  660.087006      kPEX             [S_Br]   
570       keller4        4  657.474286      kPEX             [S_Br]   
684  wiki_talk_en        2  651.742897      kPEX         [Pivot_Br]   

     time_ratio  
646    2.718623  
491    2.679591  
470    1.255654  
570    3.077975  
684    1.066777  

Error Analysis:
Average time ratio: 24.87x slower
Worst time ratio: 6700.00x slower
Best time ratio: 1.00x slower
Median time gap: 2.10 seconds
Largest time gap: 1797.10 seconds
Smallest time gap: 0.10 seconds

Error Severity Distribution:
time_ratio
≤5%         80
5–20%      114
20–100%     86
>100%       95
Name: count, dtype: int64

Algorithm‐wise Errors:
  UMKP: 0 errors, avg

#### LR with L2 regularization

In [354]:
# Instantiate and train
clf_l2_strat_y_K = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    batch_size=128,       # Feeding whole data at a time.
    epochs=1000,
)
clf_l2_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, epochs=1000, lr=0.25, penalty='l2')

Coefficients (Linear Models)

In [355]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_l2_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.426148
1           d_avg     0.309265
2       shell_num     0.206892
3        core_min     0.205416
4           d_max     0.203007
5        core_max     0.197777
6   shell_maxSize     0.108241
7               n     0.108043
8   shell_minSize     0.096344
9         k_value     0.091241
10  shell_stdSize     0.062822
11  shell_avgSize     0.060956


#### Inference on 20% dataset

In [356]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.33%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 56.00% (25 predictions)
  kPEX: 60.26% (302 predictions)
  Pivot_Br: 25.00% (44 predictions)
  S_Br: 34.69% (49 predictions)

Recall by Algorithm:
  UMKP: 32.08% (17/53 cases)
  AltRB: 44.93% (31/69 cases)
  kPEX: 87.44% (195/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 50.68% (74/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55062.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.187
  Avg Overhead/case ((pred - opt) / opt)    = 134.86%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 196 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [358]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_l2_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.66%

Precision by Algorithm:
  UMKP: 0.00% (0 predictions)
  AltRB: 50.82% (61 predictions)
  kPEX: 56.31% (547 predictions)
  Pivot_Br: 23.75% (80 predictions)
  S_Br: 37.50% (72 predictions)

Recall by Algorithm:
  UMKP: 22.00% (22/100 cases)
  AltRB: 42.24% (49/116 cases)
  kPEX: 90.63% (329/363 cases)
  Pivot_Br: 58.85% (133/226 cases)
  S_Br: 43.75% (112/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 101316.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.515
  Avg Overhead/case ((pred - opt) / opt)    = 278.40%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 375 wrong predictions with time metrics

T

#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

In [359]:
# Instantiate and train
clf_elasticnet_strat_y_K = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.6,         # regularization strength of L1 in usint elasticnet
    lr=0.5,              # Learning rate
    batch_size=128,
    epochs=1000,
)
clf_elasticnet_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, beta=0.6, epochs=1000, lr=0.5,
                        penalty='elasticnet')

Coefficients (Linear Models)

In [360]:
feature_cols = X_train.columns.tolist()
coef_df = torch_logistic_feature_importances(clf_elasticnet_strat_y_K, feature_cols)
print(coef_df)

          feature  coefficents
0               m     0.359199
1       shell_num     0.198811
2           d_max     0.178036
3   shell_minSize     0.100727
4           d_avg     0.100040
5        core_min     0.080962
6         k_value     0.075069
7   shell_stdSize     0.064561
8        core_max     0.063770
9               n     0.053538
10  shell_maxSize     0.053468
11  shell_avgSize     0.025339


#### Inference on 20% dataset

In [361]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 51.19%

Precision by Algorithm:
  UMKP: 16.67% (12 predictions)
  AltRB: 44.44% (9 predictions)
  kPEX: 58.02% (293 predictions)
  Pivot_Br: 33.33% (54 predictions)
  S_Br: 40.38% (52 predictions)

Recall by Algorithm:
  UMKP: 35.85% (19/53 cases)
  AltRB: 30.43% (21/69 cases)
  kPEX: 85.20% (190/223 cases)
  Pivot_Br: 65.81% (102/155 cases)
  S_Br: 49.32% (72/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54890.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.183
  Avg Overhead/case ((pred - opt) / opt)    = 113.38%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 205 wrong predictions with time metrics

Top 

#### Re-evalute on entire dataset

In [362]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_elasticnet_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LR_results/tie_tol/50/L1+L2/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.53%

Precision by Algorithm:
  UMKP: 40.00% (15 predictions)
  AltRB: 56.25% (32 predictions)
  kPEX: 55.43% (534 predictions)
  Pivot_Br: 27.00% (100 predictions)
  S_Br: 46.84% (79 predictions)

Recall by Algorithm:
  UMKP: 28.00% (28/100 cases)
  AltRB: 31.90% (37/116 cases)
  kPEX: 89.26% (324/363 cases)
  Pivot_Br: 60.62% (137/226 cases)
  S_Br: 45.70% (117/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99995.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.495
  Avg Overhead/case ((pred - opt) / opt)    = 267.07%


Adding prediction probabilities...

Saving full predictions to './LR_results/tie_tol/50/L1+L2/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 376 wrong predictions with time metri

Metrics written to ./LR_results/tie_tol/50/L1+L2/split_and_expand/model_performance_for_all_experiments.xlsx
